In [1]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

import numpy as np
import pandas as pd
from os import listdir
from os.path import isfile, join
from mhcnames import normalize_allele_name
import mhcnames
import mhcgnomes


# 1. Read data

In [2]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
DATA_PATH = f'{DATA_PATH_BASE}/core_identification/' #Work laptop

# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu

In [3]:
onlyfiles = [f for f in listdir(DATA_PATH) if isfile(join(DATA_PATH, f))]
ALLELES = []
NOT_NORMALIZED = []
per_allele_train_df = dict()
for file_name in onlyfiles:
    allele_name = file_name.split("_", 1)[1].split('.')[0]
    NOT_NORMALIZED.append(allele_name)
    allele_name = mhcgnomes.parse(allele_name).to_string()
    per_allele_train_df[allele_name] =  pd.read_csv(DATA_PATH + file_name,sep=";")
    ALLELES.append(allele_name)
ALLELES

['HLA-DQA1*05:01/DQB1*02:01', 'HLA-DPA1*02:02/DPB1*05:01']

In [4]:
mhcnames.compact_allele_name('HLA-DPA1*01:03-DPB1*04:01')

'DPA10103-DPB10401'

# 2. Make per length split

In [5]:
from sklearn.model_selection import KFold
MIN_LENGTH = 8
MAX_LENGTH = 30

#leave only binders
def getBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 1]['peptide'].values
    return result_peptides
def getNonBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 0]['peptide'].values
    return result_peptides

def make_per_length_dict(df, binder_flag):
    lengths = set(df['length'])
    lengths = set([length for length in lengths if length >= MIN_LENGTH and length <= MAX_LENGTH])
    per_length_dict = dict()
    for length in lengths:
        result_peptides = df[(df['binder'] == binder_flag) & (df['length'] == length)]['peptide'].values
        per_length_dict[length] = result_peptides
    return per_length_dict

kf = KFold(n_splits=5, shuffle=True)
def make_per_length_per_KFold_dict(per_length_dict):

    per_length_per_kfold_peptides_train = dict()
    per_length_per_kfold_peptides_test = dict()

    print("Length \t Train/Test for splits", end="")
    for target_length in per_length_dict.keys():
        print(f"\n{target_length}:", end="\t")
        peptides_array = per_length_dict[target_length]
        per_length_per_kfold_peptides_train[target_length] = dict()
        per_length_per_kfold_peptides_test[target_length] = dict()
        if len(peptides_array) > kf.n_splits:
            i = 0
            for train_indexes_b, test_indexes_b in kf.split(peptides_array):
                print("{}/{}\t".format(len(train_indexes_b), len(test_indexes_b)), end="")
                current_train = peptides_array[train_indexes_b]
                current_test = peptides_array[test_indexes_b]
                per_length_per_kfold_peptides_train[target_length][i] = current_train
                per_length_per_kfold_peptides_test[target_length][i] = current_test
                i = i + 1
        else:

            print(f"Not enough peptides of length {target_length}", end="")
    print()
    return per_length_per_kfold_peptides_train , per_length_per_kfold_peptides_test





def addAuxColumns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        df['length'] = df['peptide'].str.len()

per_allele_binders_train = dict()
per_allele_non_binders_train = dict()
per_allele_per_length_binders_train = dict()
per_allele_per_length_non_binders_train = dict()
per_allele_per_length_per_kfold_binders_train = dict()
per_allele_per_length_per_kfold_binders_test = dict()
per_allele_per_length_per_kfold_non_binders_train = dict()
per_allele_per_length_per_kfold_non_binders_test = dict()


for allele_name in ALLELES:
    print(f"Allele {allele_name}")
    df_train = per_allele_train_df[allele_name]
    df_train = df_train.sample(frac=1).reset_index(drop=True)
    per_allele_binders_train[allele_name] = getBindersPeptidesOnly(df_train)
    per_allele_non_binders_train[allele_name] = getNonBindersPeptidesOnly(df_train)
    per_allele_per_length_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=1)
    per_allele_per_length_non_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=0)
    print(f"Binders")
    per_allele_per_length_per_kfold_binders_train[allele_name],  per_allele_per_length_per_kfold_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_binders_train[allele_name])
    print(f"Non-binders")
    per_allele_per_length_per_kfold_non_binders_train[allele_name],  per_allele_per_length_per_kfold_non_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_non_binders_train[allele_name])


Allele HLA-DQA1*05:01/DQB1*02:01
Binders
Length 	 Train/Test for splits
8:	13/4	13/4	14/3	14/3	14/3	
9:	91/23	91/23	91/23	91/23	92/22	
10:	87/22	87/22	87/22	87/22	88/21	
11:	152/39	153/38	153/38	153/38	153/38	
12:	277/70	277/70	278/69	278/69	278/69	
13:	442/111	442/111	442/111	443/110	443/110	
14:	575/144	575/144	575/144	575/144	576/143	
15:	520/130	520/130	520/130	520/130	520/130	
16:	456/114	456/114	456/114	456/114	456/114	
17:	329/83	329/83	330/82	330/82	330/82	
18:	156/40	157/39	157/39	157/39	157/39	
19:	107/27	107/27	107/27	107/27	108/26	
20:	75/19	75/19	75/19	75/19	76/18	
21:	55/14	55/14	55/14	55/14	56/13	
22:	34/9	34/9	34/9	35/8	35/8	
23:	26/7	26/7	26/7	27/6	27/6	
24:	18/5	18/5	18/5	19/4	19/4	
25:	14/4	14/4	14/4	15/3	15/3	
27:	Not enough peptides of length 27
Non-binders
Length 	 Train/Test for splits
8:	Not enough peptides of length 8
9:	Not enough peptides of length 9
10:	Not enough peptides of length 10
11:	Not enough peptides of length 11
12:	Not enough peptides of length 12

# 3. Save per_length_per_split to files

In [6]:
import os
from collections import defaultdict

TARGET_PATH_TO_FILES = DATA_PATH  + '/per_length_per_kfold_split/'
if not os.path.exists(TARGET_PATH_TO_FILES):
    os.makedirs(TARGET_PATH_TO_FILES)

def save_data(per_allele_per_length_per_kfold_data, data_type=None, dataset_type=None):
    assert data_type in ["binders", "nonbinders"]
    assert dataset_type in ["train", "test"]

    per_allele_dfs = defaultdict(list)
    for allele_name in per_allele_per_length_per_kfold_data.keys():
        for length, per_k_fold_dict in per_allele_per_length_per_kfold_data[allele_name].items():
                for split_num, peptides in per_k_fold_dict.items():
                    df = pd.DataFrame({"peptide":peptides, "length":length, "split": split_num})
                    per_allele_dfs[allele_name].append(df)
        total_df= pd.concat(per_allele_dfs[allele_name])
        target_allele_name = mhcnames.compact_allele_name(allele_name)
        total_df.to_csv(f'{TARGET_PATH_TO_FILES}{target_allele_name}_{data_type}_{dataset_type}.csv',sep=";", index=False)


save_data(per_allele_per_length_per_kfold_binders_train, "binders", "train")
# save_data(per_allele_per_length_per_kfold_non_binders_train, "nonbinders", "train")
save_data(per_allele_per_length_per_kfold_binders_test, "binders", "test")
# save_data(per_allele_per_length_per_kfold_non_binders_test, "nonbinders", "test")